# Taobao CTR: 3-tower co-train vs. strong baselines (full data, GPU)

Click-through-rate (pCTR) prediction on the real Alibaba Taobao display-ads data, using **all** of it:
1.14M users, 26.6M ad impressions and the full 723M-row user behaviour log.

| # | Model | Idea |
|---|---|---|
| 1 | LightGBM | Gradient-boosted trees on click-rate features + user–ad interaction counts |
| 2 | MLP | Embeddings of user, context and ad IDs → MLP |
| 3 | MLP + sequence | Adds the average of the user's last 50 actions |
| 4 | Transformer | Self-attention over the last 50 actions + the target ad (BST-style) |
| 5 | Two-tower | User tower · ad tower dot product (fast; user and ad never meet) |
| 6 | **3-tower co-train** | Two-tower + a cross tower (the ad attends over the user's history) + inter module. One model, two scores: a cheap two-tower score and a rich 3-tower score (Pinterest, 2026) |

**Pipeline:** full data → features → each model trained with early stopping on a validation set → Optuna tuning →
comparison on the test day (last day) → combining models (Pinterest-style cascade, blend) → MLflow tracking and model registry.

**Runs on Google Colab or on Kaggle** (the notebook detects which one).

*On Colab*
1. Runtime → Change runtime type → **T4 GPU**.
2. Runtime → Run all, and approve Google Drive access when asked. Caches, predictions, results and MLflow runs are kept in `MyDrive/taobao_final/`.
3. The notebook is **resumable**: every finished step is saved to Drive, so if Colab disconnects, reconnect and Run all again; finished steps are loaded instead of redone.

*On Kaggle*
1. Add the dataset: **Add Input → search "taobao-ad-display-click" (by zerotrustai)**. It is read directly from `/kaggle/input`, so nothing is downloaded.
2. Settings: **Accelerator → GPU T4 x2** (or P100), **Internet → On** (needed to install MLflow; requires a phone-verified account).
3. **Save Version → Save & Run All (Commit)**: it runs in the background, so you can close the browser. Results, models and MLflow runs land in the version's Output (`/kaggle/working/taobao_final/`).

The first full run takes about 4–5 hours (building the data takes about 25 min of that).

## 1. Setup

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:            # Google Drive keeps caches, results and MLflow runs, so a disconnected run can resume
    from google.colab import drive
    drive.mount("/content/drive")
packages = ["mlflow", "optuna", "lightgbm"] + ([] if ON_KAGGLE else ["kagglehub"])
subprocess.run([sys.executable, "-m", "pip", "-q", "install", *packages], check=False)
try:
    import mlflow
except ImportError:
    raise SystemExit("MLflow could not be installed. On Kaggle: Settings > Internet > On (needs a phone-verified account), then rerun.")
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local")

In [ ]:
import os, gc, json, time, glob, ctypes, threading, queue, numbers, shutil, re
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import pyarrow as pa, pyarrow.parquet as pq
import lightgbm as lgb, optuna, mlflow
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU (Colab: Runtime > Change runtime type > T4 GPU; Kaggle: Settings > Accelerator > GPU). "
          "The data build and LightGBM run fine on CPU; the deep models would take days.")

def free_memory():
    "Give unused RAM and GPU memory back."
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. Configuration
Everything you may want to change is here. The default training settings are the ones that worked best
in development (plain BCE, Adam, lr 1e-3, batch 4096); early stopping picks the best checkpoint.

In [ ]:
CFG = dict(
    # where things live
    # caches, predictions, results, MLflow: Drive on Colab (survives disconnects), the version's Output on Kaggle
    drive_dir="/kaggle/working/taobao_final" if ON_KAGGLE else "/content/drive/MyDrive/taobao_final",
    local_dir="/tmp/data" if ON_KAGGLE else "/content/data",   # fast local disk: raw CSVs and temporary buckets
    kaggle_dataset="zerotrustai/taobao-ad-display-click",

    # data
    seq_len=50,          # last N actions before each impression
    n_buckets=32,        # the behaviour log is split into user buckets to keep RAM low
    chunk_rows=10_000_000,
    te_alpha=20,         # smoothing of the click-rate (target) encodings

    # model sizes (same for every deep model)
    emb_dim=16, tower_dim=64, n_heads=4, n_query_tokens=4,

    # default training settings
    lr=1e-3, batch_size=4096, dropout=0.1, optimizer="adam", weight_decay=0.0,
    loss="bce", gamma=2.0, pos_weight=1.0, cotrain_weight=0.5,
    max_epochs=2,                 # upper limit; early stopping keeps the best checkpoint
    eval_every_rows=4_000_000,    # check validation AUC every ~4M training rows
    patience=3,                   # stop after 3 checks without improvement

    # Optuna tuning
    tune_models=["3-tower co-train", "Two-tower", "MLP", "MLP + sequence", "Transformer"],
    tune_rows=3_000_000,          # trials train on a random 3M training rows for 1 epoch...
    tune_val_rows=1_000_000,      # ...and are scored on 1M validation rows
    tune_trials=15,               # trials per model (upper limit)
    tune_timeout_min=12,          # or stop a model's search after this many minutes

    # combining models
    cascade_top_frac=0.4,         # Pinterest re-scores the top candidates with the 3-tower; here the top 40%

    seed=42,
)

D = CFG["drive_dir"]
for sub in ["", "cache", "preds", "models"]:
    os.makedirs(os.path.join(D, sub), exist_ok=True)
os.makedirs(CFG["local_dir"], exist_ok=True)
set_seed(CFG["seed"])

# results of every run, saved after each model so the notebook can resume
RESULTS_FILE = os.path.join(D, "results.json")
RESULTS = json.load(open(RESULTS_FILE)) if os.path.exists(RESULTS_FILE) else {}

def save_result(key, row):
    RESULTS[key] = row
    json.dump(RESULTS, open(RESULTS_FILE, "w"), indent=1, default=float)

def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

## 3. MLflow experiment tracking
Every training run logs its settings, validation curve, test metrics and model weights.
The tracking database lives on local disk (SQLite does not work well on Google Drive) and is copied to Drive after each run.
To browse runs later on your own computer: download `mlflow.db` and `mlartifacts/` from `MyDrive/taobao_final/` and run
`mlflow ui --backend-store-uri sqlite:///mlflow.db`.

In [ ]:
MLFLOW_DB = os.path.join(CFG["local_dir"], "mlflow.db")
MLFLOW_DB_DRIVE = os.path.join(D, "mlflow.db")
if os.path.exists(MLFLOW_DB_DRIVE) and not os.path.exists(MLFLOW_DB):
    shutil.copy(MLFLOW_DB_DRIVE, MLFLOW_DB)          # resume an earlier run's history

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
if mlflow.get_experiment_by_name("taobao-ctr") is None:
    mlflow.create_experiment("taobao-ctr", artifact_location=os.path.join(D, "mlartifacts"))
mlflow.set_experiment("taobao-ctr")

def backup_mlflow():
    shutil.copy(MLFLOW_DB, MLFLOW_DB_DRIVE)

print("MLflow experiment ready; runs so far:", len(mlflow.search_runs()))

## 4. Data

| File | Rows | One row = |
|---|---|---|
| `raw_sample.csv` | 26.6M | one ad shown to one user (label `clk` = clicked) |
| `ad_feature.csv` | 0.85M | one ad: category, campaign, advertiser, brand, price |
| `user_profile.csv` | 1.06M | one user: segment, gender, age, spending power, city tier... |
| `behavior_log.csv` | 723M (24 GB) | one shopping action: view / cart / favourite / buy of a category + brand |

**Fitting 723M actions in 12.7 GB of RAM.** Each impression needs the user's last 50 actions *before* it. Storing that
as a 26.6M × 50 table would take ~12 GB, so instead:
1. Stream the behaviour CSV once, writing it into 32 user buckets on local disk.
2. For each bucket, sort by (user, time) and keep only actions that can be in some impression's last-50 history
   (≈215M of the 723M). Each impression just stores two pointers into that compact store.
3. Batches gather their histories on the fly from the store.

Everything is cached on Drive, so later runs load it in a minute or two.

In [ ]:
USER_COLS = ["cms_segid", "cms_group_id", "final_gender_code", "age_level",
             "pvalue_level", "shopping_level", "occupation", "new_user_class_level"]
CTX_COLS = ["pid", "hour"]                                   # ad slot, local hour of day
AD_ID_COLS = ["adgroup_id", "campaign_id", "customer"]
ID_COLS = USER_COLS + CTX_COLS + AD_ID_COLS + ["ad_cate", "ad_brand", "price_bucket"]
TE_COLS = ["te_user", "te_ad", "te_campaign", "te_customer", "te_brand", "te_cate", "te_user_cate", "log_price"]
BTAG = {"pv": 1, "cart": 2, "fav": 3, "buy": 4}             # action types (0 = padding)
L = CFG["seq_len"]

def kaggle_csv(fname, retries=4):
    "Local path of one Kaggle file. Downloads it the first time, retrying if the connection breaks."
    if ON_KAGGLE:                                                     # dataset attached as an input: no download
        hits = [p for p in glob.glob(f"/kaggle/input/**/{fname}", recursive=True) if os.path.isfile(p)]
        if not hits:
            raise FileNotFoundError(f"{fname} not found. Add Input > search 'taobao-ad-display-click' (zerotrustai).")
        return hits[0]
    folder = os.path.join(CFG["local_dir"], fname.replace(".csv", ""))
    complete = os.path.join(folder, ".complete")                     # written only after a full download
    hits = [p for p in glob.glob(f"{folder}/**/{fname}", recursive=True) if os.path.isfile(p)]
    if hits and os.path.exists(complete):
        return hits[0]
    import kagglehub
    for attempt in range(1, retries + 1):
        try:                                                          # force: never reuse a half-downloaded file
            path = kagglehub.dataset_download(CFG["kaggle_dataset"], path=f"{fname}/{fname}",
                                              output_dir=folder, force_download=True)
            open(complete, "w").close()
            return path
        except Exception as e:                                        # e.g. "Connection broken: IncompleteRead"
            print(f"  download of {fname} failed (attempt {attempt}/{retries}): {type(e).__name__}")
            if attempt == retries:
                raise
            time.sleep(30 * attempt)                                  # wait a bit longer each time

def encode(raw, vocab):
    "Raw ids -> 1..K using a sorted array of known ids; unknown or missing -> 0."
    raw = np.asarray(raw)
    pos = np.clip(np.searchsorted(vocab, raw), 0, len(vocab) - 1)
    return np.where((raw >= 0) & (vocab[pos] == raw), pos + 1, 0)

### 4.1 Impressions, ad and user features

In [ ]:
def load_impressions():
    raw = pd.read_csv(kaggle_csv("raw_sample.csv"), usecols=["user", "time_stamp", "adgroup_id", "pid", "clk"],
                      dtype={"user": "int32", "time_stamp": "int64", "adgroup_id": "int32", "pid": "category", "clk": "int8"})
    raw = raw.sort_values("time_stamp", kind="stable").reset_index(drop=True)      # time order for the split
    ads = pd.read_csv(kaggle_csv("ad_feature.csv")).set_index("adgroup_id")
    users = pd.read_csv(kaggle_csv("user_profile.csv"))
    users.columns = users.columns.str.strip()                                        # one column name has a trailing space
    users = users.rename(columns={"userid": "user"}).drop_duplicates("user").set_index("user")

    ts = raw["time_stamp"].values
    user_ids, u = np.unique(raw["user"].values, return_inverse=True)
    u = u.ravel().astype(np.int64) + 1                                               # users 1..U
    ad_row = ads.index.get_indexer(raw["adgroup_id"].values)                         # -1 = unknown ad
    user_row = users.index.get_indexer(raw["user"].values)                           # -1 = no profile

    def lookup(table, rows, col):                                                    # one column per impression (NaN = missing)
        return np.where(rows >= 0, table[col].values.astype("float64")[rows], np.nan)

    X, sizes = {}, {}
    columns = [(c, lookup(users, user_row, c)) for c in USER_COLS] + \
              [(c, lookup(ads, ad_row, c)) for c in ["campaign_id", "customer"]] + \
              [("pid", raw["pid"].cat.codes.values.astype("float64")),
               ("hour", (((ts + 8 * 3600) // 3600) % 24).astype("float64")),          # China is UTC+8
               ("adgroup_id", raw["adgroup_id"].values.astype("float64"))]
    for c, values in columns:
        vocab = np.unique(values[~np.isnan(values)])
        X[c] = encode(values, vocab).astype(np.int32)
        sizes[c] = len(vocab)

    price = np.nan_to_num(lookup(ads, ad_row, "price"))
    X["price_bucket"] = (np.clip(np.floor(np.log1p(price) * 2), 0, 38) + 1).astype(np.int32)
    print(f"impressions {len(raw):,} | users {len(user_ids):,} | CTR {raw['clk'].mean():.4f}")
    return dict(ts=ts, y=raw["clk"].values.astype(np.float32), u=u, user_ids=user_ids, X=X, sizes=sizes, price=price,
                ad_cate=lookup(ads, ad_row, "cate_id"), ad_brand=lookup(ads, ad_row, "brand"))

### 4.2 Stream the behaviour log into user buckets

In [ ]:
BUCKET_DIR = os.path.join(CFG["local_dir"], "buckets")

def stream_behaviour(imp):
    "One pass over the 24 GB CSV; rows go to n_buckets Parquet files by user. Returns the distinct categories/brands seen."
    done = os.path.join(BUCKET_DIR, "done.npz")
    if os.path.exists(done):
        return np.load(done)
    os.makedirs(BUCKET_DIR, exist_ok=True)
    NB, t0 = CFG["n_buckets"], time.time()
    t_lo, t_hi = imp["ts"].min() - 30 * 86_400, imp["ts"].max()        # the file has some corrupt timestamps
    user_index = pd.Index(imp["user_ids"])
    writers, cates, brands, n_read = [None] * NB, [], [], 0
    reader = pd.read_csv(kaggle_csv("behavior_log.csv"), chunksize=CFG["chunk_rows"],
                         usecols=["user", "time_stamp", "btag", "cate", "brand"],
                         dtype={"user": "int32", "time_stamp": "int64", "btag": "category", "cate": "int32", "brand": "float64"})
    for i, ch in enumerate(reader):
        n_read += len(ch)
        ch = ch[ch["time_stamp"].between(t_lo, t_hi)]
        u = user_index.get_indexer(ch["user"].values)
        keep = u >= 0                                                       # users who saw at least one ad
        ch, u = ch[keep], (u[keep] + 1).astype(np.int32)
        cate = ch["cate"].values.astype(np.int64)
        brand = ch["brand"].fillna(-1).values.astype(np.int64)
        cates.append(np.unique(cate)); brands.append(np.unique(brand))
        bucket = u % NB
        order = np.argsort(bucket, kind="stable")
        table = pa.table({"u": u[order], "t": ch["time_stamp"].values[order],
                          "b": ch["btag"].astype(str).map(BTAG).fillna(0).astype("int8").values[order],
                          "c": cate[order], "r": brand[order]})
        cuts = np.searchsorted(bucket[order], np.arange(NB + 1))
        for k in range(NB):
            if writers[k] is None:
                writers[k] = pq.ParquetWriter(os.path.join(BUCKET_DIR, f"b{k}.parquet"), table.schema)
            writers[k].write_table(table.slice(cuts[k], cuts[k + 1] - cuts[k]))
        if i % 10 == 0:
            print(f"  {n_read:,} rows read | {(time.time() - t0) / 60:.1f} min")
    for w in writers:
        w.close()
    np.savez(done, cate=np.unique(np.concatenate(cates)), brand=np.unique(np.concatenate(brands)))
    print(f"streamed {n_read:,} behaviour rows in {(time.time() - t0) / 60:.1f} min")
    return np.load(done)

### 4.3 Compact history store

In [ ]:
def build_history(imp, cate_vocab, brand_vocab):
    "Per bucket: sort by (user, time), keep only actions that can appear in a last-L history, record pointers."
    u_imp, ts, NB = imp["u"], imp["ts"], CFG["n_buckets"]
    U = len(imp["user_ids"]) + 1
    first_t, last_t = np.zeros(U, np.int64), np.zeros(U, np.int64)
    us, idx = np.unique(u_imp, return_index=True);        first_t[us] = ts[idx]          # ts is sorted
    us, idx = np.unique(u_imp[::-1], return_index=True);  last_t[us] = ts[::-1][idx]

    first, end = np.zeros(len(ts), np.int64), np.zeros(len(ts), np.int64)
    parts, offset = {"BC": [], "BB": [], "BT": []}, 0
    cdtype = np.int16 if len(cate_vocab) < 32_000 else np.int32
    imp_bucket = u_imp % NB
    for k in range(NB):
        d = pq.read_table(os.path.join(BUCKET_DIR, f"b{k}.parquet"))
        u, t = d["u"].to_numpy().astype(np.int64), d["t"].to_numpy()
        order = np.lexsort((t, u)); u, t = u[order], t[order]
        key = (u << 32) + t                                                     # sortable (user, time)
        us = np.unique(u)
        start_u = np.searchsorted(u, us)
        end_first = np.searchsorted(key, (us << 32) + first_t[us])              # history end at the user's first impression
        end_last = np.searchsorted(key, (us << 32) + last_t[us])                # ... at the last impression
        lo = np.maximum(start_u, end_first - L)                                 # nothing older than this is ever used
        n = end_last - lo
        new_start = offset + np.concatenate([[0], np.cumsum(n)[:-1]])
        keep = np.repeat(lo - (new_start - offset), n) + np.arange(n.sum())      # rows to keep, in sorted order
        parts["BC"].append(encode(d["c"].to_numpy()[order][keep], cate_vocab).astype(cdtype))
        parts["BB"].append(encode(d["r"].to_numpy()[order][keep], brand_vocab).astype(np.int32))
        parts["BT"].append(d["b"].to_numpy()[order][keep].astype(np.int8))

        rows = np.where(imp_bucket == k)[0]                                     # this bucket's impressions
        iu = u_imp[rows]
        j = np.clip(np.searchsorted(us, iu), 0, len(us) - 1)
        has = us[j] == iu                                                       # user has any behaviour?
        e = np.searchsorted(key, (iu << 32) + ts[rows])                         # only actions BEFORE the impression
        end[rows] = np.where(has, e - lo[j] + new_start[j], 0)
        first[rows] = np.where(has, new_start[j], 0)
        offset += int(n.sum())
        del d, u, t, order, key, keep
        gc.collect()

    def join(chunks):                                                           # concatenate without doubling RAM
        out, p = np.empty(sum(len(c) for c in chunks), chunks[0].dtype), 0
        while chunks:
            c = chunks.pop(0); out[p:p + len(c)] = c; p += len(c)
        return out
    BC, BB, BT = join(parts["BC"]), join(parts["BB"]), join(parts["BT"])
    print(f"history store: {len(BT):,} actions ({(BC.nbytes + BB.nbytes + BT.nbytes) / 1e9:.1f} GB)")
    return first, end, BC, BB, BT

### 4.4 Click-rate features (time-ordered)
Smoothed past click rate of the user, ad, campaign, advertiser, brand, category and user×category, plus log price.

**Time-ordered, like CatBoost's ordered target statistics:** a training row's click rate uses only clicks from *earlier*
rows, so a row never sees its own label. Validation rows use the statistics known at the end of the fit period, and
the test day uses all training days. That way validation is encoded exactly like the test day.

*Why not out-of-fold (as in the development notebook)?* Leaving a row's fold out makes its encoded rate slightly
*lower* when the row was clicked (its own click is missing), so on the training rows click rates become negatively
related to clicks, while the test day, encoded from all training rows, has the normal positive relation. Models learn
the wrong direction and validation, encoded the same way as training, does not notice. Development runs showed a
validation–test AUC gap of about 0.025 partly for this reason.

In [ ]:
def click_rate_features(imp, X, fit_rows, val_rows, test_rows, n_cate):
    y = imp["y"]
    prior, a = y[fit_rows].mean(), CFG["te_alpha"]
    train_rows = np.concatenate([fit_rows, val_rows])

    def rate_table(keys, rows):
        g = pd.DataFrame({"k": keys[rows], "y": y[rows]}).groupby("k")["y"].agg(["sum", "count"])
        return (g["sum"] + a * prior) / (g["count"] + a)

    def target_encode(keys):
        out = np.empty(len(keys), np.float32)
        # fit rows (time-sorted): clicks and views of the same key in EARLIER rows only
        k, yy = keys[fit_rows], y[fit_rows]
        g = pd.DataFrame({"k": k, "y": yy}).groupby("k")["y"]
        clicks_before, views_before = g.cumsum().values - yy, g.cumcount().values
        out[fit_rows] = (clicks_before + a * prior) / (views_before + a)
        # validation: what was known at the end of the fit period; test: what was known at the end of training
        out[val_rows] = pd.Series(keys[val_rows]).map(rate_table(keys, fit_rows)).fillna(prior).values
        out[test_rows] = pd.Series(keys[test_rows]).map(rate_table(keys, train_rows)).fillna(prior).values
        return out

    u = imp["u"]
    te = np.stack([target_encode(u), target_encode(X["adgroup_id"]), target_encode(X["campaign_id"]),
                   target_encode(X["customer"]), target_encode(X["ad_brand"]), target_encode(X["ad_cate"]),
                   target_encode(u * (n_cate + 1) + X["ad_cate"]),
                   np.log1p(imp["price"]).astype(np.float32)], 1)
    mu, sd = te[fit_rows].mean(0), te[fit_rows].std(0) + 1e-6
    return ((te - mu) / sd).astype(np.float32)                       # standardised with fit-period statistics

### 4.5 Build everything once (or load the cache)

In [ ]:
CACHE = os.path.join(D, "cache")
ARRAYS = ["y", "ts", "first", "end", "BC", "BB", "BT", "TE"] + ["x_" + c for c in ID_COLS]

def split(ts):
    "Last day = test; the last 10% of the training days (by time) = validation."
    test_start = ts.max() - 86_400
    train_rows, test_rows = np.where(ts < test_start)[0], np.where(ts >= test_start)[0]
    n_val = int(0.1 * len(train_rows))
    return train_rows, train_rows[:-n_val], train_rows[-n_val:], test_rows

if not os.path.exists(os.path.join(CACHE, "sizes.json")):
    t0 = time.time()
    imp = load_impressions()
    ad_cate = np.nan_to_num(imp.pop("ad_cate"), nan=-1).astype(np.int64)
    ad_brand = np.nan_to_num(imp.pop("ad_brand"), nan=-1).astype(np.int64)
    HIST, VOCAB = os.path.join(CACHE, "history.npz"), os.path.join(CACHE, "vocab.npz")
    if os.path.exists(HIST) and os.path.exists(VOCAB):              # checkpoint: skip the 30-min download + stream
        v = np.load(VOCAB)
        cate_vocab, brand_vocab = v["cate"], v["brand"]
    else:
        seen = stream_behaviour(imp)
        cate_vocab = np.unique(np.concatenate([ad_cate[ad_cate >= 0], seen["cate"]]))      # shared by ads and history
        brand_vocab = np.unique(np.concatenate([ad_brand[ad_brand >= 0], seen["brand"][seen["brand"] >= 0]]))
    X, sizes = imp["X"], imp["sizes"]
    X["ad_cate"], X["ad_brand"] = encode(ad_cate, cate_vocab).astype(np.int32), encode(ad_brand, brand_vocab).astype(np.int32)
    sizes["cate"], sizes["brand"] = len(cate_vocab), len(brand_vocab)
    del ad_cate, ad_brand; free_memory()

    if os.path.exists(HIST) and os.path.exists(VOCAB):
        h = np.load(HIST)
        first, end, BC, BB, BT = (h[k] for k in ["first", "end", "BC", "BB", "BT"])
        print("loaded the history-store checkpoint from Drive")
    else:
        first, end, BC, BB, BT = build_history(imp, cate_vocab, brand_vocab)
        np.savez(HIST, first=first, end=end, BC=BC, BB=BB, BT=BT)   # checkpoint on Drive (survives a disconnect)
        np.savez(VOCAB, cate=cate_vocab, brand=brand_vocab)
        print(f"history-store checkpoint saved | {(time.time() - t0) / 60:.1f} min")
    _, fit, val, test = split(imp["ts"])
    TE = click_rate_features(imp, X, fit, val, test, len(cate_vocab))
    arrays = dict(y=imp["y"], ts=imp["ts"], first=first, end=end, BC=BC, BB=BB, BT=BT, TE=TE,
                  **{"x_" + c: X[c] for c in ID_COLS})
    for k, v in arrays.items():
        np.save(os.path.join(CACHE, k + ".npy"), v)
    json.dump({k: int(v) for k, v in sizes.items()}, open(os.path.join(CACHE, "sizes.json"), "w"))
    print(f"data built and cached in {(time.time() - t0) / 60:.1f} min")
    del imp, arrays, X, first, end, BC, BB, BT, TE
    free_memory()

A = {k: np.load(os.path.join(CACHE, k + ".npy")) for k in ARRAYS}       # load the cache (fresh, compact arrays)
y, ts, first, end, BC, BB, BT, TE = (A.pop(k) for k in ["y", "ts", "first", "end", "BC", "BB", "BT", "TE"])
X = {c: A.pop("x_" + c) for c in ID_COLS}
SIZES = json.load(open(os.path.join(CACHE, "sizes.json")))
SIZES.update(price_bucket=39, btag=4)
train_rows, fit_rows, val_rows, test_rows = split(ts)
free_memory()
print(f"fit {len(fit_rows):,} | validation {len(val_rows):,} | test {len(test_rows):,} | test CTR {y[test_rows].mean():.4f}")
print(f"history store {len(BT):,} actions")

## 5. Batches
Each batch gathers its users' last 50 actions from the compact store. A background thread builds the next batches on the
CPU while the GPU trains on the current one.

In [ ]:
AR = np.arange(L)

def history(rows):
    "Last L actions before each impression: categories, brands, action types (0 = padding)."
    idx = end[rows, None] - L + AR[None, :]
    ok = idx >= first[rows, None]
    idx = np.where(ok, idx, 0)
    return np.where(ok, BC[idx], 0), np.where(ok, BB[idx], 0), np.where(ok, BT[idx], 0)

def make_batch(rows):
    b = {c: torch.from_numpy(X[c][rows].astype(np.int64)) for c in ID_COLS}
    hc, hb, ht = history(rows)
    b["hist_cate"], b["hist_brand"], b["hist_btag"] = (torch.from_numpy(h.astype(np.int64)) for h in (hc, hb, ht))
    b["te"] = torch.from_numpy(TE[rows])
    b["label"] = torch.from_numpy(y[rows])
    return b

def batches(rows, batch_size, shuffle=False):
    "Yields batches on the GPU, built in a background thread."
    if shuffle:
        rows = np.random.permutation(rows)
    q, stop = queue.Queue(maxsize=6), threading.Event()

    def producer():
        for s in range(0, len(rows), batch_size):
            if stop.is_set():
                return
            q.put(make_batch(rows[s:s + batch_size]))
        q.put(None)

    threading.Thread(target=producer, daemon=True).start()
    try:
        while (b := q.get()) is not None:
            yield {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}
    finally:                                   # early stop: let the producer finish
        stop.set()
        while not q.empty():
            q.get_nowait()

## 6. Models
All deep models share the same embedding tables and click-rate features. The click-rate features are split by side:
user-only rates may go into a user tower, ad-only rates into an ad tower, and the user×category rate (which needs both sides)
only into models where user and ad meet.

In [ ]:
TE_USER, TE_AD, TE_CROSS = [0], [1, 2, 3, 4, 5, 7], [6]          # columns of TE (see TE_COLS)
USER_FEATS = [(c, c) for c in USER_COLS + CTX_COLS]               # (batch key, embedding table)
AD_FEATS = [("adgroup_id", "adgroup_id"), ("campaign_id", "campaign_id"), ("customer", "customer"),
            ("ad_cate", "cate"), ("ad_brand", "brand"), ("price_bucket", "price_bucket")]

def mlp(n_in, hidden, n_out, dropout):
    layers = []
    for h in hidden:
        layers += [nn.Linear(n_in, h), nn.ReLU(), nn.Dropout(dropout)]
        n_in = h
    return nn.Sequential(*layers, nn.Linear(n_in, n_out))

class Embeddings(nn.Module):
    "One table per ID feature; category and brand tables are shared by the ad and the history."
    def __init__(self, d):
        super().__init__()
        self.t = nn.ModuleDict({k: nn.Embedding(n + 1, d, padding_idx=0) for k, n in SIZES.items()})
    def ids(self, b, feats):
        return torch.cat([self.t[table](b[key]) for key, table in feats], -1)
    def history(self, b):                                         # (B, L, d): one vector per past action
        return self.t["cate"](b["hist_cate"]) + self.t["brand"](b["hist_brand"]) + self.t["btag"](b["hist_btag"])
    def mean_history(self, b):                                    # average of the real (non-padding) actions
        h, m = self.history(b), (b["hist_btag"] > 0).float().unsqueeze(-1)
        return (h * m).sum(1) / m.sum(1).clamp(min=1)

N_USER, N_AD = len(USER_FEATS), len(AD_FEATS)

class MLP(nn.Module):
    def __init__(self, hp):
        super().__init__()
        d = hp["emb_dim"]
        self.emb = Embeddings(d)
        self.net = mlp((N_USER + N_AD) * d + 8, [256, 128, 64], 1, hp["dropout"])
    def forward(self, b):
        x = torch.cat([self.emb.ids(b, USER_FEATS), self.emb.ids(b, AD_FEATS), b["te"]], -1)
        return {"logit": self.net(x).squeeze(-1)}

class MLPSequence(nn.Module):
    def __init__(self, hp):
        super().__init__()
        d = hp["emb_dim"]
        self.emb = Embeddings(d)
        self.net = mlp((N_USER + N_AD + 1) * d + 8, [256, 128, 64], 1, hp["dropout"])
    def forward(self, b):
        x = torch.cat([self.emb.ids(b, USER_FEATS), self.emb.ids(b, AD_FEATS), self.emb.mean_history(b), b["te"]], -1)
        return {"logit": self.net(x).squeeze(-1)}

class Transformer(nn.Module):
    "History tokens + the target ad as the last token, through self-attention (BST-style)."
    def __init__(self, hp):
        super().__init__()
        d, D = hp["emb_dim"], hp["tower_dim"]
        self.emb = Embeddings(d)
        self.tok = nn.Linear(d, D)
        self.pos = nn.Embedding(L + 1, D)
        layer = nn.TransformerEncoderLayer(D, hp["n_heads"], 2 * D, dropout=hp["dropout"], batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)
        self.net = mlp((N_USER + N_AD) * d + 2 * D + 8, [256, 128, 64], 1, hp["dropout"])
    def forward(self, b):
        target = self.emb.t["cate"](b["ad_cate"]) + self.emb.t["brand"](b["ad_brand"])     # ad in the same space as actions
        x = self.tok(torch.cat([self.emb.history(b), target.unsqueeze(1)], 1)) + self.pos.weight
        pad = torch.cat([b["hist_btag"] == 0, torch.zeros_like(b["hist_btag"][:, :1], dtype=torch.bool)], 1)
        z = self.enc(x, src_key_padding_mask=pad)
        keep = (~pad).float().unsqueeze(-1)
        mean = (z * keep).sum(1) / keep.sum(1)
        x = torch.cat([self.emb.ids(b, USER_FEATS), self.emb.ids(b, AD_FEATS), z[:, -1], mean, b["te"]], -1)
        return {"logit": self.net(x).squeeze(-1)}

class UserTower(nn.Module):
    "User side only: profile + context + averaged history + user click rate."
    def __init__(self, emb, hp):
        super().__init__()
        self.emb = emb
        self.net = mlp((N_USER + 1) * hp["emb_dim"] + len(TE_USER), [256, 128], hp["tower_dim"], hp["dropout"])
    def forward(self, b):
        return self.net(torch.cat([self.emb.ids(b, USER_FEATS), self.emb.mean_history(b), b["te"][:, TE_USER]], -1))

class AdTower(nn.Module):
    "Ad side only (Pinterest's 'Pin tower'): can be computed offline and cached per ad."
    def __init__(self, emb, hp):
        super().__init__()
        self.emb = emb
        n = N_AD * hp["emb_dim"] + len(TE_AD)
        self.net = mlp(n, [256, 128], hp["tower_dim"], hp["dropout"])
        self.to_cross = nn.Linear(n, hp["tower_dim"])            # what the ad 'asks' the cross tower (also cacheable)
    def forward(self, b):
        x = torch.cat([self.emb.ids(b, AD_FEATS), b["te"][:, TE_AD]], -1)
        return self.net(x), self.to_cross(x)

class TwoTower(nn.Module):
    def __init__(self, hp):
        super().__init__()
        self.emb = Embeddings(hp["emb_dim"])
        self.user, self.ad = UserTower(self.emb, hp), AdTower(self.emb, hp)
        self.bias = nn.Parameter(torch.zeros(1))
    def forward(self, b):
        return {"logit": (self.user(b) * self.ad(b)[0]).sum(-1) + self.bias}

class CrossTower(nn.Module):
    "The ad's query tokens attend over the user's history: this is where user and ad meet."
    def __init__(self, emb, hp):
        super().__init__()
        D, self.nq = hp["tower_dim"], hp["n_query_tokens"]
        self.emb = emb
        self.proj, self.pos, self.norm = nn.Linear(hp["emb_dim"], D), nn.Embedding(L, D), nn.LayerNorm(D)
        self.queries = nn.Linear(D, self.nq * D)
        self.empty = nn.Parameter(torch.zeros(1, 1, D))          # always-present token (users with no history)
        self.attn = nn.MultiheadAttention(D, hp["n_heads"], batch_first=True)
        self.out = nn.Linear(D, D)
    def forward(self, ad_query, b):
        B = ad_query.shape[0]
        k = self.norm(self.proj(self.emb.history(b)) + self.pos.weight)
        k = torch.cat([self.empty.expand(B, 1, -1), k], 1)
        pad = torch.cat([torch.zeros_like(b["hist_btag"][:, :1], dtype=torch.bool), b["hist_btag"] == 0], 1)
        q = self.queries(ad_query).view(B, self.nq, -1)
        a, _ = self.attn(q, k, k, key_padding_mask=pad)
        return self.out(a.mean(1))

class ThreeTower(nn.Module):
    "Two-tower + cross tower + inter module. Two heads: rich 3-tower score and cheap two-tower score (co-trained)."
    def __init__(self, hp):
        super().__init__()
        D = hp["tower_dim"]
        self.emb = Embeddings(hp["emb_dim"])
        self.user, self.ad, self.cross = UserTower(self.emb, hp), AdTower(self.emb, hp), CrossTower(self.emb, hp)
        self.inter = mlp(3 * D + len(TE_CROSS), [128, 64], 1, hp["dropout"])
        self.task_bias = nn.Linear(2 * D, 1)
        self.dot_bias = nn.Parameter(torch.zeros(1))
    def forward(self, b):
        u = self.user(b)
        a, a_query = self.ad(b)
        c = self.cross(a_query, b)
        rich = self.inter(torch.cat([u, a, c, b["te"][:, TE_CROSS]], -1)) + self.task_bias(torch.cat([u, a], -1))
        return {"logit": rich.squeeze(-1), "cotrain_logit": (u * a).sum(-1) + self.dot_bias}

MODELS = {"3-tower co-train": ThreeTower, "Two-tower": TwoTower, "MLP": MLP,
          "MLP + sequence": MLPSequence, "Transformer": Transformer}

## 7. Training with early stopping
Every ~4M training rows the model is scored on the validation set (the last 10% of the training days).
The best checkpoint is kept; training stops after 3 checks without improvement. The test day is used only once, at the end.

In [ ]:
def make_loss(hp):
    if hp["loss"] == "weighted_bce":                                   # clicks count pos_weight times more
        w = torch.tensor(hp["pos_weight"], device=DEVICE)
        return lambda z, t: F.binary_cross_entropy_with_logits(z, t, pos_weight=w)
    if hp["loss"] == "focal":                                          # down-weights easy examples
        def focal(z, t):
            ce = F.binary_cross_entropy_with_logits(z, t, reduction="none")
            p = torch.sigmoid(z)
            return ((1 - (p * t + (1 - p) * (1 - t))) ** hp["gamma"] * ce).mean()
        return focal
    return lambda z, t: F.binary_cross_entropy_with_logits(z, t)

def make_optimizer(model, hp):
    opt = {"adam": torch.optim.Adam, "adamw": torch.optim.AdamW, "adagrad": torch.optim.Adagrad}[hp["optimizer"]]
    return opt(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])

@torch.no_grad()
def predict(model, rows):
    "Click probabilities of every head for the given rows."
    model.eval()
    out = {}
    for b in batches(rows, 16384):
        for k, v in model(b).items():
            out.setdefault(k, []).append(torch.sigmoid(v).float().cpu().numpy())
    return {k: np.concatenate(v) for k, v in out.items()}

def metrics(labels, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return {"AUC": roc_auc_score(labels, p), "LogLoss": log_loss(labels, p), "PR_AUC": average_precision_score(labels, p)}

def train_model(model_cls, hp, fit, val, max_epochs, eval_every_rows, patience, report=None, verbose=True):
    "Train, check validation AUC regularly, keep the best checkpoint. report(i, auc) lets Optuna prune a trial."
    set_seed(CFG["seed"])
    model = model_cls(hp).to(DEVICE)
    opt, loss_fn = make_optimizer(model, hp), make_loss(hp)
    eval_every = max(1, eval_every_rows // hp["batch_size"])
    best = {"auc": -1.0, "step": 0, "state": None}
    curve, step, bad, t0 = [], 0, 0, time.time()

    def check():
        nonlocal bad
        auc = roc_auc_score(y[val], predict(model, val)["logit"])
        model.train()
        curve.append((step, auc))
        if auc > best["auc"] + 1e-4:
            best.update(auc=auc, step=step, state={k: v.detach().clone() for k, v in model.state_dict().items()})
            bad = 0
        else:
            bad += 1
        if verbose:
            print(f"    step {step:>6}: val AUC {auc:.4f} (best {best['auc']:.4f}) | {(time.time() - t0) / 60:.1f} min")
        if report:
            report(len(curve), auc)                 # may raise optuna.TrialPruned
        return bad >= patience

    model.train()
    stop = False
    for epoch in range(max_epochs):
        for b in batches(fit, hp["batch_size"], shuffle=True):
            out = model(b)
            loss = loss_fn(out["logit"], b["label"])
            if "cotrain_logit" in out:                                  # 3-tower: also train the cheap head
                loss = loss + hp["cotrain_weight"] * loss_fn(out["cotrain_logit"], b["label"])
            opt.zero_grad()
            loss.backward()
            opt.step()
            step += 1
            if step % eval_every == 0 and check():
                stop = True
                break
        if stop:
            break
        if step % eval_every != 0 and check():                         # also check at the end of every epoch
            break
    model.load_state_dict(best["state"])
    return model, best, curve

### Run one model and record everything
`run_model` trains a model, scores validation and test, saves its predictions and weights to Drive, and logs the run to MLflow.
A model that has already finished is skipped, which is what makes the notebook resumable.

In [ ]:
HP_KEYS = ["lr", "batch_size", "dropout", "optimizer", "weight_decay", "loss", "gamma", "pos_weight", "cotrain_weight"]

def pred_file(key):
    return os.path.join(D, "preds", slug(key) + ".npz")

def run_model(name, hp, variant):
    key = f"{name} [{variant}]"
    if key in RESULTS and os.path.exists(pred_file(key)):
        print(f"{key}: already done (val AUC {RESULTS[key]['val_AUC']:.4f}, test AUC {RESULTS[key]['AUC']:.4f})")
        return
    print(f"=== {key} ===")
    t0 = time.time()
    with mlflow.start_run(run_name=key):
        mlflow.log_params({"model": name, "variant": variant, **{k: hp[k] for k in HP_KEYS}})
        model, best, curve = train_model(MODELS[name], hp, fit_rows, val_rows,
                                         hp["max_epochs"], hp["eval_every_rows"], hp["patience"])
        for step, auc in curve:
            mlflow.log_metric("val_auc", auc, step=step)
        p_val, p_test = predict(model, val_rows), predict(model, test_rows)
        np.savez(pred_file(key), **{f"{h}_val": v for h, v in p_val.items()}, **{f"{h}_test": v for h, v in p_test.items()})
        torch.save(model.state_dict(), os.path.join(D, "models", slug(key) + ".pt"))
        row = {"model": name, "variant": variant, "val_AUC": best["auc"], "best_step": best["step"],
               **metrics(y[test_rows], p_test["logit"]), "train_min": (time.time() - t0) / 60,
               "params_M": sum(p.numel() for p in model.parameters()) / 1e6, "hp": {k: hp[k] for k in HP_KEYS}}
        if "cotrain_logit" in p_test:                                    # the cheap head of the same model
            row["head2_AUC"] = roc_auc_score(y[test_rows], p_test["cotrain_logit"])
        mlflow.log_metrics({k: row[k] for k in ["val_AUC", "AUC", "LogLoss", "PR_AUC", "train_min"] + (["head2_AUC"] if "head2_AUC" in row else [])})
    save_result(key, row)
    backup_mlflow()
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items() if k != "hp"})
    del model
    free_memory()

DEFAULT_HP = {k: CFG[k] for k in ["emb_dim", "tower_dim", "n_heads", "n_query_tokens", "max_epochs",
                                  "eval_every_rows", "patience"] + HP_KEYS}

## 8. LightGBM (the strongest classic baseline)
Same features and settings as the development notebook: the 8 click-rate features, 4 user–ad interaction counts
(how often the user recently touched the ad's category / brand) and 9 small categoricals. Rows are fed to LightGBM in
batches (`lgb.Sequence`), so the 21M × 21 feature matrix never sits in RAM at once.

In [ ]:
LOWCARD = USER_COLS[1:] + CTX_COLS
LGB_FEATURES = TE_COLS + ["n_same_cate", "n_same_brand", "n_same_brand_strong", "hist_len"] + LOWCARD

def lgb_features(rows):
    hc, hb, ht = history(rows)
    ac, ab = X["ad_cate"][rows, None], X["ad_brand"][rows, None]
    same_cate = (hc == ac) & (ac > 0)
    same_brand = (hb == ab) & (ab > 0)
    inter = np.stack([same_cate.sum(1), same_brand.sum(1), (same_brand & (ht > 1)).sum(1), (ht > 0).sum(1)], 1)
    cats = np.stack([X[c][rows] for c in LOWCARD], 1)
    return np.hstack([TE[rows], inter, cats]).astype(np.float64)    # LightGBM's Sequence input must be float64

class FeatureSequence(lgb.Sequence):
    batch_size = 500_000
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        if isinstance(i, numbers.Integral):
            return lgb_features(self.rows[[i]])[0]
        return lgb_features(self.rows[i])

def lgb_predict(model, rows):
    return np.concatenate([model.predict(lgb_features(rows[s:s + 500_000]), num_iteration=model.best_iteration)
                           for s in range(0, len(rows), 500_000)])

KEY = "LightGBM [default]"
if KEY in RESULTS and os.path.exists(pred_file(KEY)):
    print(f"{KEY}: already done (test AUC {RESULTS[KEY]['AUC']:.4f})")
else:
    t0 = time.time()
    params = {"objective": "binary", "learning_rate": 0.05, "num_leaves": 63, "min_data_in_leaf": 200,
              "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
              "metric": "auc", "verbose": -1, "seed": CFG["seed"]}
    with mlflow.start_run(run_name=KEY):
        mlflow.log_params({"model": "LightGBM", "variant": "default", **params})
        d_fit = lgb.Dataset(FeatureSequence(fit_rows), y[fit_rows], feature_name=LGB_FEATURES,
                            categorical_feature=LOWCARD, free_raw_data=True)
        d_val = lgb.Dataset(FeatureSequence(val_rows), y[val_rows], reference=d_fit)
        gbm = lgb.train(params, d_fit, num_boost_round=1000, valid_sets=[d_val],
                        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)])
        del d_fit, d_val; free_memory()
        p_val, p_test = lgb_predict(gbm, val_rows), lgb_predict(gbm, test_rows)
        np.savez(pred_file(KEY), logit_val=p_val, logit_test=p_test)
        gbm.save_model(os.path.join(D, "models", "lightgbm.txt"))
        row = {"model": "LightGBM", "variant": "default", "val_AUC": roc_auc_score(y[val_rows], p_val),
               "best_step": gbm.best_iteration, **metrics(y[test_rows], p_test), "train_min": (time.time() - t0) / 60}
        mlflow.log_metrics({k: row[k] for k in ["val_AUC", "AUC", "LogLoss", "PR_AUC", "train_min"]})
        mlflow.log_artifact(os.path.join(D, "models", "lightgbm.txt"))
        importance = pd.Series(gbm.feature_importance("gain"), index=LGB_FEATURES).sort_values(ascending=False)
    save_result(KEY, row)
    backup_mlflow()
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
    print(importance.head(10).round(0))
    del gbm; free_memory()

## 9. Deep models with the default settings
The 3-tower goes first (it is the main model), the Transformer last (it is the slowest, roughly 15 min per epoch on a T4).

In [ ]:
for name in MODELS:
    run_model(name, DEFAULT_HP, "default")

## 10. Fair tuning with Optuna
Each deep model gets its own search. Lessons from development are built in:
* **Trials** train on a random 3M training rows for 1 epoch and are scored on 1M validation rows (the test day is never used).
* **Trial 0 is the default setting**; the TPE sampler proposes the rest and the median pruner stops clearly worse trials early.
* **The search space is kept sane:** `pos_weight` ≤ 3 (a tuned `pos_weight` of 9.4 broke the 3-tower in development: good
  on validation, much worse on the test day), learning rates around the default.
* The best setting is refit on all training rows with early stopping, and it **replaces the default only if its validation
  AUC is higher**.

| Hyper-parameter | Search range |
|---|---|
| Learning rate | 3e-4 to 3e-3 (log) |
| Dropout | 0.0 to 0.3 |
| Optimizer | Adam, AdamW, Adagrad |
| Weight decay | 1e-7 to 1e-4 (log) |
| Batch size | 2048, 4096, 8192 |
| Loss | BCE, focal (gamma 0.5–2), weighted BCE (pos_weight 1–3) |
| Co-train weight (3-tower only) | 0.25 to 1.0 |

In [ ]:
OPTUNA_DB = os.path.join(CFG["local_dir"], "optuna.db")
OPTUNA_DB_DRIVE = os.path.join(D, "optuna.db")
if os.path.exists(OPTUNA_DB_DRIVE) and not os.path.exists(OPTUNA_DB):
    shutil.copy(OPTUNA_DB_DRIVE, OPTUNA_DB)                            # resume earlier studies

def hp_from_params(params, name):
    "Turn Optuna parameters into a full settings dict."
    hp = dict(DEFAULT_HP)
    hp.update({k: v for k, v in params.items() if k in HP_KEYS})
    return hp

def suggest(trial, name):
    params = {
        "lr": trial.suggest_float("lr", 3e-4, 3e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "optimizer": trial.suggest_categorical("optimizer", ["adam", "adamw", "adagrad"]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-7, 1e-4, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [2048, 4096, 8192]),
        "loss": trial.suggest_categorical("loss", ["bce", "focal", "weighted_bce"]),
    }
    if params["loss"] == "focal":
        params["gamma"] = trial.suggest_float("gamma", 0.5, 2.0)
    if params["loss"] == "weighted_bce":
        params["pos_weight"] = trial.suggest_float("pos_weight", 1.0, 3.0)
    if name == "3-tower co-train":
        params["cotrain_weight"] = trial.suggest_float("cotrain_weight", 0.25, 1.0)
    return hp_from_params(params, name)

DEFAULT_TRIAL = {"lr": 1e-3, "dropout": 0.1, "optimizer": "adam", "weight_decay": 1e-7,
                 "batch_size": 4096, "loss": "bce", "cotrain_weight": 0.5}

rng = np.random.default_rng(CFG["seed"])
tune_fit = np.sort(rng.choice(fit_rows, min(CFG["tune_rows"], len(fit_rows)), replace=False))
tune_val = np.sort(rng.choice(val_rows, min(CFG["tune_val_rows"], len(val_rows)), replace=False))

def tune(name):
    study = optuna.create_study(study_name=slug(name), storage=f"sqlite:///{OPTUNA_DB}", load_if_exists=True,
                                direction="maximize", sampler=optuna.samplers.TPESampler(seed=CFG["seed"]),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=3))
    finished = [t for t in study.trials if t.state in (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)]
    if len(finished) >= CFG["tune_trials"]:
        print(f"{name}: tuning already done ({len(finished)} trials, best val AUC {study.best_value:.4f})")
        return study
    if not study.trials:
        study.enqueue_trial({k: v for k, v in DEFAULT_TRIAL.items() if name == "3-tower co-train" or k != "cotrain_weight"})

    def objective(trial):
        hp = suggest(trial, name)
        def report(i, auc):
            trial.report(auc, i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        with mlflow.start_run(run_name=f"{name} trial {trial.number}", nested=True):
            mlflow.log_params(trial.params)
            eval_rows = max(len(tune_fit) // 2, hp["batch_size"])            # 2 checks per trial (for pruning)
            model, best, _ = train_model(MODELS[name], hp, tune_fit, tune_val, 1, eval_rows, 99, report, verbose=False)
            mlflow.log_metric("val_auc", best["auc"])
        del model
        free_memory()
        return best["auc"]

    t0 = time.time()
    with mlflow.start_run(run_name=f"{name} [optuna]"):
        study.optimize(objective, n_trials=CFG["tune_trials"] - len(finished),
                       timeout=CFG["tune_timeout_min"] * 60, catch=(RuntimeError,))
        mlflow.log_params({"best_" + k: v for k, v in study.best_params.items()})
        mlflow.log_metric("best_trial_val_auc", study.best_value)
    shutil.copy(OPTUNA_DB, OPTUNA_DB_DRIVE)
    backup_mlflow()
    states = pd.Series([t.state.name for t in study.trials]).value_counts().to_dict()
    print(f"{name}: best trial val AUC {study.best_value:.4f} (default trial {study.trials[0].value}) | {states} | "
          f"{(time.time() - t0) / 60:.1f} min | {study.best_params}")
    return study

STUDIES = {name: tune(name) for name in CFG["tune_models"]}

### Refit the best setting of each model on all training rows

In [ ]:
for name, study in STUDIES.items():
    if study.best_trial.number == 0:                     # trial 0 is the default setting
        print(f"{name}: the default setting won the search; nothing to refit")
        continue
    run_model(name, hp_from_params(study.best_params, name), "tuned")

## 11. Results
For every model, the variant (default or tuned) with the higher **validation** AUC is selected; the table shows its
test-day metrics. AUC = ranking quality (higher is better), LogLoss = probability quality (lower is better),
PR-AUC = precision–recall area (only ~5% of impressions are clicked).

In [ ]:
res = pd.DataFrame(RESULTS.values())
res["selected"] = res["val_AUC"] == res.groupby("model")["val_AUC"].transform("max")
cols = ["model", "variant", "val_AUC", "AUC", "LogLoss", "PR_AUC", "head2_AUC", "best_step", "train_min", "params_M"]
res = res.reindex(columns=cols + ["selected", "hp"])
display(res[cols + ["selected"]].sort_values(["selected", "AUC"], ascending=False).round(4).reset_index(drop=True))

final = res[res["selected"]].sort_values("AUC")
fig, ax = plt.subplots(figsize=(8, 0.5 * len(final) + 1))
ax.barh(final["model"], final["AUC"], color=["#DD8452" if "3-tower" in m else "#4C72B0" for m in final["model"]])
ax.set_xlim(final["AUC"].min() - 0.005, final["AUC"].max() + 0.003)
for i, v in enumerate(final["AUC"]):
    ax.text(v + 0.0003, i, f"{v:.4f}", va="center", fontsize=9)
ax.set_xlabel("Test AUC (last day, full data)")
ax.set_title("Taobao CTR: selected variant of each model")
plt.tight_layout(); plt.show()
res[cols].to_csv(os.path.join(D, "results.csv"), index=False)

## 12. Combining models
**How Pinterest serves the 3-tower** (engineering blog, *Beyond Two Towers*, Part 2): the cheap two-tower head scores every
candidate (often 100K+ per request), the 3-tower re-scores only the top 40K by that score, and the rest keep their
two-tower score. Here:
* **Cascade:** a stage-1 model ranks all ads; the top K% are re-scored by the 3-tower; the rest keep the stage-1 score.
  Tried with the two-tower head (Pinterest) and with LightGBM as stage 1. K = 40%, and the K that is best on validation.
* **Blend:** a weighted average of LightGBM and 3-tower (in logit space), weight chosen on validation.

Note: Taobao has no candidate lists per request, so the cascade ranks all ads of the day together. In production the main
point of a cascade is speed; offline it shows how much accuracy the cheap first stage costs.

In [ ]:
def selected_preds(model):
    key = res[(res["model"] == model) & res["selected"]].iloc[0]
    return np.load(pred_file(f"{model} [{key['variant']}]"))

p3, pg = selected_preds("3-tower co-train"), selected_preds("LightGBM")
P = {"3-tower": (p3["logit_val"], p3["logit_test"]),
     "two-tower head": (p3["cotrain_logit_val"], p3["cotrain_logit_test"]),
     "LightGBM": (pg["logit_val"], pg["logit_test"])}
y_val, y_test = y[val_rows], y[test_rows]

def cascade(stage1, stage2, frac):
    out = stage1.copy()
    top = np.argsort(-stage1)[:int(frac * len(stage1))]
    out[top] = stage2[top]
    return out

def blend(pa_, pb_, w):
    logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
    return 1 / (1 + np.exp(-(w * logit(pa_) + (1 - w) * logit(pb_))))

rows = [{"method": f"{m} alone", **metrics(y_test, P[m][1])} for m in P]
sweep = []
for stage1 in ["two-tower head", "LightGBM"]:
    for frac in [0.1, 0.2, 0.4, 0.6, 0.8]:
        sweep.append({"stage 1": stage1, "top_frac": frac,
                      "val_AUC": roc_auc_score(y_val, cascade(P[stage1][0], P["3-tower"][0], frac)),
                      "test_AUC": roc_auc_score(y_test, cascade(P[stage1][1], P["3-tower"][1], frac))})
    k = CFG["cascade_top_frac"]
    rows.append({"method": f"{stage1} -> 3-tower on top {k:.0%}", **metrics(y_test, cascade(P[stage1][1], P["3-tower"][1], k))})
    s = pd.DataFrame(sweep); kb = s[s["stage 1"] == stage1].sort_values("val_AUC").iloc[-1]["top_frac"]
    rows.append({"method": f"{stage1} -> 3-tower on top {kb:.0%} (val-chosen)", **metrics(y_test, cascade(P[stage1][1], P["3-tower"][1], kb))})
W = np.linspace(0, 1, 21)
wb = W[np.argmax([roc_auc_score(y_val, blend(P["LightGBM"][0], P["3-tower"][0], w)) for w in W])]
rows.append({"method": f"blend {wb:.2f} x LightGBM + {1 - wb:.2f} x 3-tower (val-chosen)",
             **metrics(y_test, blend(P["LightGBM"][1], P["3-tower"][1], wb))})

combos = pd.DataFrame(rows).set_index("method").sort_values("AUC", ascending=False)
display(combos.round(4))
display(pd.DataFrame(sweep).pivot(index="top_frac", columns="stage 1", values="test_AUC").round(4))
combos.to_csv(os.path.join(D, "combinations.csv"))
with mlflow.start_run(run_name="combinations"):
    mlflow.log_metrics({slug(m)[:250]: v for m, v in combos["AUC"].items()})
    mlflow.log_artifact(os.path.join(D, "combinations.csv"))
backup_mlflow()

## 13. MLOps: register the best 3-tower model
The selected 3-tower is logged as a PyTorch model and registered in the MLflow **model registry** as `taobao-ctr-3tower`,
with the alias `champion`. A serving job can then load "the current champion" without knowing which run produced it:
`mlflow.pytorch.load_model("models:/taobao-ctr-3tower@champion")`.

In [ ]:
row = res[(res["model"] == "3-tower co-train") & res["selected"]].iloc[0]
key = f"3-tower co-train [{row['variant']}]"
best_model = ThreeTower(hp_from_params(row["hp"], "3-tower co-train"))
best_model.load_state_dict(torch.load(os.path.join(D, "models", slug(key) + ".pt"), map_location="cpu"))

def register_champion(model, key, row):
    with mlflow.start_run(run_name="register 3-tower"):
        mlflow.log_params({"source": key, **row["hp"]})
        mlflow.log_metrics({"test_AUC": row["AUC"], "val_AUC": row["val_AUC"],
                            "test_LogLoss": row["LogLoss"], "test_PR_AUC": row["PR_AUC"]})
        try:
            try:      # recent MLflow: pickle format (its default traced format needs example inputs)
                mlflow.pytorch.log_model(model, name="model", registered_model_name="taobao-ctr-3tower",
                                         serialization_format="pickle")
            except TypeError:                    # older MLflow
                mlflow.pytorch.log_model(model, "model", registered_model_name="taobao-ctr-3tower")
            client = mlflow.MlflowClient()
            version = max(int(v.version) for v in client.search_model_versions("name='taobao-ctr-3tower'"))
            client.set_registered_model_alias("taobao-ctr-3tower", "champion", version)
            json.dump({"source": key, "version": version}, open(os.path.join(D, "registered.json"), "w"))
            print(f"registered taobao-ctr-3tower version {version} as @champion")
        except Exception as e:                   # the weights are still saved on Drive either way
            print("model registry step failed:", e)
            mlflow.log_artifact(os.path.join(D, "models", slug(key) + ".pt"))
    backup_mlflow()

done = os.path.join(D, "registered.json")
if os.path.exists(done) and json.load(open(done)).get("source") == key:
    print(f"{key} is already registered as taobao-ctr-3tower@champion")
else:
    register_champion(best_model, key, row)

runs = mlflow.search_runs(filter_string="tags.mlflow.parentRunId IS NULL")
show = [c for c in ["tags.mlflow.runName", "metrics.val_AUC", "metrics.AUC", "metrics.LogLoss", "metrics.PR_AUC", "metrics.train_min"] if c in runs]
display(runs[show].sort_values("metrics.AUC", ascending=False).round(4).reset_index(drop=True))

## 14. Reading the results

**Data size matters most for the deep models.** In development, the same 3-tower went from 0.628 test AUC on a 100K-user
sample to 0.640 on the full data; LightGBM's advantage on small data (click-rate features carry most of the signal)
largely disappears at full scale. (Development used out-of-fold click-rate encodings; this notebook uses time-ordered
ones, see 4.4, so its numbers are not directly comparable and validation should now track the test day closely.)

**Why the 3-tower is the model to deploy, even if LightGBM is close or slightly ahead on AUC:**
* **Serving scale.** Ads lightweight ranking scores 10K–100K+ candidates per request within tens of milliseconds.
  The ad tower's output can be computed offline and cached; the user tower runs once per request. A GBDT instead needs
  user × ad features (user×category click rate, history matches) computed for every candidate pair.
* **User–ad interaction is learned, not hand-built.** The cross tower lets the ad attend over the user's last 50 actions;
  LightGBM gets the same signal only from engineered features and target-encoding pipelines that go stale.
* **Two scores from one model.** The co-trained two-tower head is a free, cheap stage-1 score for the cascade.
* Pinterest reports ~30% lower offline loss than their previous production model, about +1% CTR online and ~1% lower
  cost-per-click, for a modest GPU cost.

**Honest caveats.** The Taobao data has one label (click), no candidate lists and no latency budget, so this notebook can
show accuracy, not the serving speed-up. Small AUC differences (< ~0.002) are within run-to-run noise.

**Next steps:** LightGBM tuning; a second seed for error bars; latency benchmark of cached-ad-tower scoring vs.
LightGBM feature computation; adding conversion labels for multi-task heads.